<a href="https://colab.research.google.com/github/vibecode66/pytorch/blob/main/Train_LLM_AutoResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/karpathy/autoresearch.git /root/autoresearch

Cloning into '/root/autoresearch'...
remote: Enumerating objects: 193, done.
remote: Total 193 (delta 0), reused 0 (delta 0), pack-reused 193 (from 1)
Receiving objects: 100% (193/193), 534.87 KiB | 25.47 MiB/s, done.
Resolving deltas: 100% (98/98), done.


In [4]:
%cd /root/autoresearch

/root/autoresearch


In [5]:
!ls

analysis.ipynb	program.md    pyproject.toml  train.py
prepare.py	progress.png  README.md       uv.lock


In [8]:
!pip install uv

In [6]:
!uv sync

Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Resolved 74 packages in 16ms
Prepared 67 packages in 1m 20s
Installed 67 packages in 695ms
 + annotated-doc==0.0.4
 + anyio==4.12.1
 + certifi==2026.2.25
 + charset-normalizer==3.4.4
 + click==8.3.1
 + contourpy==1.3.2
 + cycler==0.12.1
 + exceptiongroup==1.3.1
 + filelock==3.24.3
 + fonttools==4.61.1
 + fsspec==2026.2.0
 + h11==0.16.0
 + hf-xet==1.3.1
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.5.0
 + idna==3.11
 + jinja2==3.1.6
 + kernels==0.12.1
 + kiwisolver==1.4.9
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + mdurl==0.1.2
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==1.13.1.3
 + nvidia-curand-cu12==10.3.9.90
 + nvidia-c

In [7]:
!uv run prepare.py

Cache directory: /root/.cache/autoresearch

Data: downloading 11 shards (0 already exist)...
  Downloaded shard_00002.parquet
  Downloaded shard_00003.parquet
  Downloaded shard_00000.parquet
  Downloaded shard_00005.parquet
  Downloaded shard_00001.parquet
  Downloaded shard_00006.parquet
  Downloaded shard_00004.parquet
  Downloaded shard_00008.parquet
  Downloaded shard_00007.parquet
  Downloaded shard_06542.parquet
  Downloaded shard_00009.parquet
Data: 11/11 shards ready at /root/.cache/autoresearch/data

Tokenizer: training BPE tokenizer...
Tokenizer: trained in 123.2s, saved to /root/.cache/autoresearch/tokenizer/tokenizer.pkl
Tokenizer: building token_bytes lookup...
Tokenizer: saved token_bytes to /root/.cache/autoresearch/tokenizer/token_bytes.pt
Tokenizer: sanity check passed (vocab_size=8192)

Done! Ready to train.


In [12]:
!head -50 train.py

"""
Autoresearch pretraining script. Single-GPU, single-file.
Cherry-picked and simplified from nanochat.
Usage: uv run train.py
"""

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import gc
import math
import time
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from kernels import get_kernel
cap = torch.cuda.get_device_capability()
# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
fa3 = get_kernel(repo).flash_attn_interface

from prepare import MAX_SEQ_LEN, TIME_BUDGET, Tokenizer, make_dataloader, evaluate_bpb

# ---------------------------------------------------------------------------
# GPT Model
# ---------------------------------------------------------------------------

@dataclass
class GPTConfig:
    sequence_len: int = 2048
  

In [13]:
!grep -n "fa3" train.py

24:fa3 = get_kernel(repo).flash_attn_interface
93:        y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)


In [14]:
!sed -i '93s/.*/        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)/' train.py

In [15]:
!sed -i '24s/^/# /' train.py

In [20]:
!grep "n_layer\|n_embd\|vocab_size\|BATCH" train.py | head -10

    vocab_size: int = 32768
    n_layer: int = 6
    n_embd: int = 384
def has_ve(layer_idx, n_layer):
    return layer_idx % 2 == (n_layer - 1) % 2
        self.n_embd = config.n_embd
        self.head_dim = self.n_embd // self.n_head
        assert self.n_embd % self.n_head == 0
        self.c_q = nn.Linear(self.n_embd, self.n_head * self.head_dim, bias=False)
        self.c_k = nn.Linear(self.n_embd, self.n_kv_head * self.head_dim, bias=False)


In [21]:
!sed -i 's/vocab_size: int = 32768/vocab_size: int = 8192/' train.py
!sed -i 's/n_layer: int = 6/n_layer: int = 3/' train.py
!sed -i 's/n_embd: int = 384/n_embd: int = 256/' train.py

In [22]:
!grep -n "BATCH" train.py

438:TOTAL_BATCH_SIZE = 2**19 # ~524K tokens per optimizer step
451:DEVICE_BATCH_SIZE = 128  # per-device batch size (reduce if OOM)
495:tokens_per_fwdbwd = DEVICE_BATCH_SIZE * MAX_SEQ_LEN
496:assert TOTAL_BATCH_SIZE % tokens_per_fwdbwd == 0
497:grad_accum_steps = TOTAL_BATCH_SIZE // tokens_per_fwdbwd
510:train_loader = make_dataloader(tokenizer, DEVICE_BATCH_SIZE, MAX_SEQ_LEN, "train")
586:    tok_per_sec = int(TOTAL_BATCH_SIZE / dt)
587:    mfu = 100 * num_flops_per_token * TOTAL_BATCH_SIZE / dt / H100_BF16_PEAK_FLOPS
608:total_tokens = step * TOTAL_BATCH_SIZE
613:    val_bpb = evaluate_bpb(model, tokenizer, DEVICE_BATCH_SIZE)
618:steady_state_mfu = 100 * num_flops_per_token * TOTAL_BATCH_SIZE * (step - 10) / total_training_time / H100_BF16_PEAK_FLOPS if total_training_time > 0 else 0


In [23]:
!sed -i 's/DEVICE_BATCH_SIZE = 128/DEVICE_BATCH_SIZE = 16/' train.py
!sed -i 's/TOTAL_BATCH_SIZE = 2\*\*19/TOTAL_BATCH_SIZE = 2**16/' train.py

In [24]:
!uv run train.py

Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
W0613 23:20:59.040000 20654 .venv/lib/python3.10/site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode
/root/autoresearch/.venv/lib/python3.10/site-packages/torch/_inductor/lowering.py:7242: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/root/

In [34]:
!grep -n "torch.save" train.py

In [28]:
!ls -la ~/.cache/autoresearch/

total 16
drwxr-xr-x 4 root root 4096 Jun 13 22:53 .
drwxr-xr-x 1 root root 4096 Jun 13 22:56 ..
drwxr-xr-x 2 root root 4096 Jun 13 22:53 data
drwxr-xr-x 2 root root 4096 Jun 13 22:55 tokenizer


In [29]:
!find /root -name "results.tsv" 2>/dev/null

In [35]:
!sed -i '624a\    torch.save(model.state_dict(), 'model.pth')' train.py

In [45]:
!uv run train.py

Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
Traceback (most recent call last):
  File "/root/autoresearch/train.py", line 548, in <module>
    loss = model(x, y)
  File "/root/autoresearch/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py", line 414, in __call__
    return super().__call__(*args, **kwargs)
  File "/root/autoresearch/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/root/aut

In [47]:
!uv run train.py

Vocab size: 8,192
Model config: {'sequence_len': 2048, 'vocab_size': 8192, 'n_layer': 8, 'n_head': 4, 'n_kv_head': 4, 'n_embd': 512, 'window_pattern': 'SSSL'}
Parameter counts:
  wte                     : 4,194,304
  value_embeds            : 16,777,216
  lm_head                 : 4,194,304
  transformer_matrices    : 25,166,336
  scalars                 : 16
  total                   : 50,332,176
Estimated FLOPs per token: 2.390784e+08
Scaling AdamW LRs by 1/sqrt(512/768) = 1.224745
Time budget: 300s
Gradient accumulation steps: 2
W0614 00:27:39.577000 41830 .venv/lib/python3.10/site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode
/root/autoresearch/.venv/lib/python3.10/site-packages/torch/_inductor/lowering.py:7242: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/root/

In [46]:
!ls -l /root/autoresearch/model.pth

ls: cannot access '/root/autoresearch/model.pth': No such file or directory


In [41]:
!sed -i '625d' train.py

In [42]:
!sed -n '610,630p' train.py

# Final eval
model.eval()
with autocast_ctx:
    val_bpb = evaluate_bpb(model, tokenizer, DEVICE_BATCH_SIZE)

# Final summary
t_end = time.time()
startup_time = t_start_training - t_start
steady_state_mfu = 100 * num_flops_per_token * TOTAL_BATCH_SIZE * (step - 10) / total_training_time / H100_BF16_PEAK_FLOPS if total_training_time > 0 else 0
peak_vram_mb = torch.cuda.max_memory_allocated() / 1024 / 1024

print("---")
print(f"val_bpb:          {val_bpb:.6f}")
print(f"training_seconds: {total_training_time:.1f}")
print(f"total_seconds:    {t_end - t_start:.1f}")
print(f"mfu_percent:      {steady_state_mfu:.2f}")
print(f"total_tokens_M:   {total_tokens / 1e6:.1f}")
torch.save(model.state_dict(), "model.pth")
print(f"num_steps:        {step}")
print(f"num_params_M:     {num_params / 1e6:.1f}")
print(f"depth:            {DEPTH}")


In [44]:
!sed -i '627a\torch.save(model.state_dict(), "model.pth")' train.py

In [53]:
#Dowload the trained Model to local
from google.colab import files
files.download('/root/autoresearch/model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
!ls -l /root/.cache/autoresearch/data

total 985292
-rw-r--r-- 1 root root 92291918 Jun 13 22:52 shard_00000.parquet
-rw-r--r-- 1 root root 92124317 Jun 13 22:53 shard_00001.parquet
-rw-r--r-- 1 root root 91803619 Jun 13 22:52 shard_00002.parquet
-rw-r--r-- 1 root root 91793972 Jun 13 22:52 shard_00003.parquet
-rw-r--r-- 1 root root 91428652 Jun 13 22:53 shard_00004.parquet
-rw-r--r-- 1 root root 91845922 Jun 13 22:52 shard_00005.parquet
-rw-r--r-- 1 root root 91455616 Jun 13 22:53 shard_00006.parquet
-rw-r--r-- 1 root root 91202032 Jun 13 22:53 shard_00007.parquet
-rw-r--r-- 1 root root 91155507 Jun 13 22:53 shard_00008.parquet
-rw-r--r-- 1 root root 92072356 Jun 13 22:53 shard_00009.parquet
-rw-r--r-- 1 root root 91699537 Jun 13 22:53 shard_06542.parquet


In [52]:
#dowload the dataset to local
from google.colab import files
files.download('/root/.cache/autoresearch/data/shard_00000.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now that the `train.py` script has been modified to save the model, you need to run the training process again. After the training completes, a file named `model.pth` will be created in the `/root/autoresearch/` directory. You can then download this file.